# IPL Data Analysis
Notebook 1 : Data cleaning and exploration. <br>
This notebooks includes the data of ipl from the year 2008-2024. It includes the data of every match played and the data of every ball played in those matches


## 1. Setup and load data


In [38]:
import pandas as pd
matches = pd.read_csv(r"C:\Users\Parth\OneDrive\Desktop\matches.csv")
delivery = pd.read_csv(r"C:\Users\Parth\OneDrive\Desktop\deliveries.csv")
print(matches.shape)
print(delivery.shape)
print(matches.head())
print(delivery.head())

(1095, 20)
(260920, 17)
       id   season        city        date match_type player_of_match  \
0  335982  2007/08   Bangalore  2008-04-18     League     BB McCullum   
1  335983  2007/08  Chandigarh  2008-04-19     League      MEK Hussey   
2  335984  2007/08       Delhi  2008-04-19     League     MF Maharoof   
3  335985  2007/08      Mumbai  2008-04-20     League      MV Boucher   
4  335986  2007/08     Kolkata  2008-04-20     League       DJ Hussey   

                                        venue                        team1  \
0                       M Chinnaswamy Stadium  Royal Challengers Bangalore   
1  Punjab Cricket Association Stadium, Mohali              Kings XI Punjab   
2                            Feroz Shah Kotla             Delhi Daredevils   
3                            Wankhede Stadium               Mumbai Indians   
4                                Eden Gardens        Kolkata Knight Riders   

                         team2                  toss_winner toss_dec

## 2. Initial Exploration


In [39]:
print(matches.info())
print(delivery.info())
print(matches.isnull().sum())
print(delivery.isnull().sum())

<class 'pandas.DataFrame'>
RangeIndex: 1095 entries, 0 to 1094
Data columns (total 20 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id               1095 non-null   int64  
 1   season           1095 non-null   str    
 2   city             1044 non-null   str    
 3   date             1095 non-null   str    
 4   match_type       1095 non-null   str    
 5   player_of_match  1090 non-null   str    
 6   venue            1095 non-null   str    
 7   team1            1095 non-null   str    
 8   team2            1095 non-null   str    
 9   toss_winner      1095 non-null   str    
 10  toss_decision    1095 non-null   str    
 11  winner           1090 non-null   str    
 12  result           1095 non-null   str    
 13  result_margin    1076 non-null   float64
 14  target_runs      1092 non-null   float64
 15  target_overs     1092 non-null   float64
 16  super_over       1095 non-null   str    
 17  method           21 non-n

## 3. Handling Missing Values

**Matches dataset:**
- `city`: 51 nulls. Rather than filling manually, I used a
  venue → city mapping derived from existing non-null rows
  (mode of city per venue). Resolved all 51 automatically.
- `winner`: 8 nulls — these are no-result/tied matches.
  Left as null intentionally. Will filter these out in
  match analysis.
- `method`: 23 nulls — only populated when DLS was applied.
  Null = normal match. No action needed.
- `player_of_match`: Null for same no-result matches as winner.
  No action needed.

**Deliveries dataset:**
- `player_dismissed`, `fielder`, `extras_type`: Null for
  non-wicket / non-extra deliveries. This is expected —
  not missing data, just inapplicable rows. No action needed.

In [40]:
# Build a venue → city mapping from rows where city is NOT null
venue_city_map = (matches[matches['city'].notna()]
                  .groupby('venue')['city']
                  .agg(lambda x: x.mode()[0]))  # most common city for that venue

# Fill nulls using the mapping
matches['city'] = matches.apply(
    lambda row: venue_city_map.get(row['venue'], row['city'])
    if pd.isna(row['city']) else row['city'],
    axis=1
)

In [41]:
#Verifying that the nulls are gone
matches['city'].isnull().sum()

np.int64(0)

## 4. Fixing data types
Data type of 'date' should be changed to datetime data type for accurate and proper analysis. <br>
Data type of 'season' should converted to integer data type, since you can't compare string data type properly

In [42]:
#Changing data type of 'date'
matches['date'] = pd.to_datetime(matches['date'])

#Changing data type of 'season'
matches['season'] = matches['season'].str[:4].astype(int)

## 5. Fixing team name inconsistencies
Some team names got renamed over the years. I grouped the data of similar team names


In [43]:
team_name_map = {'Delhi Daredevils' : 'Delhi Capitals',
                 'Rising Pune Supergiant' : 'Rising Pune Supergiants',
                 'Kings XI Punjab' : 'Punjab Kings',
                 'Royal Challengers Bengaluru' : 'Royal Challengers Bangalore'
                 }

for col in ['team1', 'team2', 'winner', 'toss_winner']:
    matches[col] = matches[col].replace(team_name_map)

for col in ['batting_team', 'bowling_team']:
    delivery[col] = delivery[col].replace(team_name_map)

# Verifying
matches['team1'].unique()

<StringArray>
['Royal Challengers Bangalore',                'Punjab Kings',
              'Delhi Capitals',              'Mumbai Indians',
       'Kolkata Knight Riders',            'Rajasthan Royals',
             'Deccan Chargers',         'Chennai Super Kings',
        'Kochi Tuskers Kerala',               'Pune Warriors',
         'Sunrisers Hyderabad',               'Gujarat Lions',
     'Rising Pune Supergiants',        'Lucknow Super Giants',
              'Gujarat Titans']
Length: 15, dtype: str

## Creating merged dataset
Merging both the datasets to get analysis of all the data in a combined dataset. <br>
Only the necessary columns were combined from the matches dataset i.e : 'id', 'season', 'winner', 'venue', 'city', 'toss_decision', 'toss_winner', 'date'


In [44]:
merged = delivery.merge(
    matches[['id', 'season', 'winner', 'venue', 'city',
             'toss_decision', 'toss_winner', 'date']],
    left_on='match_id',
    right_on='id'
)
merged.head()


,match_id,inning,batting_team,bowling_team,over,ball,batter,bowler,non_striker,batsman_runs,...,dismissal_kind,fielder,id,season,winner,venue,city,toss_decision,toss_winner,date
0,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,1,SC Ganguly,P Kumar,BB McCullum,0,...,NaN,NaN,335982,2007,Kolkata Knight Riders,M Chinnaswamy Stadium,Bangalore,field,Royal Challengers Bangalore,2008-04-18
1,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,2,BB McCullum,P Kumar,SC Ganguly,0,...,NaN,NaN,335982,2007,Kolkata Knight Riders,M Chinnaswamy Stadium,Bangalore,field,Royal Challengers Bangalore,2008-04-18
2,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,3,BB McCullum,P Kumar,SC Ganguly,0,...,NaN,NaN,335982,2007,Kolkata Knight Riders,M Chinnaswamy Stadium,Bangalore,field,Royal Challengers Bangalore,2008-04-18
3,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,4,BB McCullum,P Kumar,SC Ganguly,0,...,NaN,NaN,335982,2007,Kolkata Knight Riders,M Chinnaswamy Stadium,Bangalore,field,Royal Challengers Bangalore,2008-04-18
4,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,5,BB McCullum,P Kumar,SC Ganguly,0,...,NaN,NaN,335982,2007,Kolkata Knight Riders,M Chinnaswamy Stadium,Bangalore,field,Royal Challengers Bangalore,2008-04-18


### Saving the new merged dataset into csv file


In [45]:
merged.to_csv('merged_data.csv', index = False)


## 8. Dataset Summary

### Final Dataset State

| Dataset | Rows | Columns | Nulls Remaining |
|---|---|---|---|
| matches | 1100 | 20 | 8 (no-result matches in `winner`) |
| deliveries | ~250,000 | 17 | Expected nulls only |
| merged | ~250,000 | 23 | Same as deliveries |

### Seasons Covered
2008 to 2024 — 17 IPL seasons

### Teams in Dataset
['Royal Challengers Bangalore',                'Punjab Kings',
              'Delhi Capitals',              'Mumbai Indians',
       'Kolkata Knight Riders',            'Rajasthan Royals',
             'Deccan Chargers',         'Chennai Super Kings',
        'Kochi Tuskers Kerala',               'Pune Warriors',
         'Sunrisers Hyderabad',               'Gujarat Lions',
     'Rising Pune Supergiants',        'Lucknow Super Giants',
              'Gujarat Titans']

### Key Cleaning Decisions
1. 51 null cities auto-filled using venue → city mapping
2. Team names standardised — 4 franchises renamed
   (Delhi, Punjab, Hyderabad, Pune)
3. Date column converted to datetime for time-series analysis
4. Merged dataset saved to data/merged.csv for use in
   subsequent notebooks

### What's Next
Notebook 2 covers match-level analysis — toss impact,
venue patterns, win margins, and Player of the Match trends.